# Introduction

Welcome to the workshop!

The goal of this part of the workshop is to get familiar with an important part of the supervised classification process with large language models. Usually the process of supervised classification with large language model can be divided into the following steps:

### Preparation

1. Decide on what to classify and make a solid definition. For example, if you want to classify a specific type of rhetoric in political transcripts, you typically find some literature, extract a definition and adapt for your purpose where it is reasonable to do so. Having a solid definition beforehand makes your life (and the life of the people who rate your test set) significantly easier.

### Which model to pick?
2. Pick an initial model. This depends on your computing resources and the difficulty of your task. There is a wide variety of models to pick from. However, make the decision easier for yourself by first deciding on:
- Do I want to run a model locally?
- Do I want to use an API?

These options have their own tradeoffs. For instance, a locally ran model needs some very powerful compute and is usually smaller than the frontier models, such as GPT-5. But it gives you more control, allows for finetuning, removes a lot of the undeterminism in your predictions (makes the predictions more stable across runs on the same data) and allows for the processing of privacy sensitive information since it does not need to go through some AI company's servers. However, it is also slower and requires a bit more technical know-how to set up.

### Make a valiation set and optimize
3. Label a validation set. With your initial definition and model, you can now randomly sample a few examples from your dataset to label by hand. This validation set will be used to set hyperparameters of the model and improve the prompt as wel will see today.

There is no strict rule for a good validation set size. Try to aim for around at least 30-40 examples per class, but since this is just a validation set it could be set lower. Since this is also not the final test set, you are also allowed to hand pick some text that you need to model to get correct, or that is very hard to label for both human or model. This would allow you to see what the model does in very specific cases and optimize for it accordingly.

4. Run the classification on the validation set and improve. This is what we will do today and this part of the workshop will go through the ins-and outs of this step.

### Test model performance
5. Label a test set and run the classification on it. A final test set is a SEPERATE randomly sampled set from your data and is used to test the performance of your classification with the intial model and the optimized prompt + hyperparameters. Here, do try to aim for at least 30-40 examples per class, do not chery-pick examples and try to run the classification once. It is also a good idea to involve multiple raters during this step and decide on one 'ground-truth' label by way of majority voting. This step could also be used to compare different models with their own 'perfected' set-ups.

### Answer your research question
6. Classify your entire dataset and run downstream analyses for whichever question you have in mind.

# Load and install required packages


We will be using the python package: choicellm for our model calls to remove a lot of code specific to initializing a model and working with a smaller model as we do here.

However, know that this can just as well be done without this package but it makes the code a bit easier to follow

In [ ]:
%pip install -qU choicellm krippendorff

In [ ]:
import pandas as pd
import json
import numpy as np
from sklearn.metrics import classification_report, roc_curve, roc_auc_score
import krippendorff
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Clone the repository

Clone the workshop repository to get the required files

It contains a baseline prompt, a validation.csv, and a potential_examples.csv with some example transcripts you could use to improve your prompt

In [ ]:
!git clone https://github.com/Kevinoost1/aps26_llm_ws.git

In [ ]:
%cd aps26_llm_ws
!git pull

# Load in the validation DataFrame

The validation dataframe contains only a hand full of labeled examples. For an actual validation set it is recommended to use a lot more

The code just extracts the manual labels and exports the actual transcripts to a .txt, which is the format we are going to use for choicellm

In [ ]:
df: pd.DataFrame = pd.read_csv('validation.csv')
y_true = df['manual_alienation']

with open('transcripts.txt', 'w', encoding='utf-8') as f:
    for t in zip(df['translation']):
        t_clean = str(t).replace('\n', ' ').replace('\r', ' ')
        f.write(t_clean + '\n')


# The Current Prompt

Running the code below, will print the starting prompt that we are going to try to improve today.

Note that this prompt has been divided into different parts for convenience:

1. The system prompt tells the model what 'role' it should take. A part of the default system prompt in the ChatGPT models is for example: 'you are a helpful assistant'. This system prompt is supplied during the entire classification process.

2. We have defined two categories for this workshop: 'Alienation' (i.e. us vs. them) rhetoric and 'Other', which is the rest category and will contain any transcripts that do not contain Alienation rhetoric.

3. Later on in this workshop you will encounter a third part of the prompt, namely examples. This turns the classification task from 'zero-shot' classification into 'few-shot' classification. This could massively improve performance if you supply the model with quality examples

4. The user prompt is made up of the categories, examples, and a new transcript for everything it needs to classify.

In [ ]:
with open('prompt_full.json') as f:
  prompt = json.load(f)

sys_prompt = prompt['system_prompt']
cat_alienation = prompt['categories']['Alienation']
cat_other = prompt['categories']['Other']

print(f"""
    System Prompt: {sys_prompt}\n
    Definition of Alienation: {cat_alienation}\n
    Definition of Other: {cat_other}\n
    """
)

# Supervised Classification

Now, the actual classification step can begin. We are using a very small toy model for this workshop that does not perform very well out-of the box.

This is because for larger models you either need some very powerful computing resources (GPUs with enough VRAM) or you need to use an API, such as the one by OPENAI. If you are able to run a powerful model locally, just replace the value for the --model argument with the model link on HuggingFace or with a local folder. If you would like to use an API, you can similarly add the --model argument to the model you would like to use (e.g. 'gpt-4o') and add the second argument --openai

For demonstration purposes, we will continue to use the small model. The validation process stays the same

In [ ]:
!choicellm transcripts.txt --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'prompt_full.json' > results.csv

# Results

With our results retrieved, we can print and inspect if there is anything odd. Likely you will get almost exclusively 'Alienation' predictions due to the model not being very good.

In [ ]:
results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']
y_prob = results['probs'].str.split(';').str[0].astype(float)

print(y_pred)

# How does the prompt perform?

We use two main ways of assesing model performance:
- F1 score -> This metric describes a balance between precision and recall

A good F1 score depends on the task, but is generally between 0.7 and 0.9.


- Krippendorff's alpha -> Generally more conservative than F1 score. For validation, try to aim for at least 0.8 at a minimum. However, if you can get it above 0.9 that is even better. For the test set you can just report the metric that comes out. If both your validation set and test set are equally representative of your total data there's a good chance that optimization on the validation set to 0.8 or 0.9 will yield good results on the test set.

Both performance metrics do not care about class imbalance (unlike accuracy). Do keep in mind that with a larger class imbalance, you need a larger test set (it is also good to have a larger validation set in that case)


--------------------------------------------------------------------------------

There is one more metric that I think is very useful, namely the receiver operator characteristic curve (ROC) and its area under the curve (AUC). I highly recommend using this metric since it bases model performance not on the majority label as predicted by the model, but rather on the uncertainty from the model (class probabilities).

However, be careful with using this metric whenever you run a model through an API. It is very likely that the probabilities you get back from the API are not the actual logits from the model, which makes this metric useless.

In [ ]:
def get_performance(y_true, y_pred, y_prob=None):
  report = classification_report(y_true, y_pred)

  le = LabelEncoder()
  le.fit(pd.concat([y_true, y_pred]))

  kripp_data = np.array([le.transform(y_true), le.transform(y_pred)])
  alpha = krippendorff.alpha(reliability_data=kripp_data, level_of_measurement='nominal')

  print(f'''
  Performance Report:\n\n
  Krippendorff Alpha: {alpha}\n
  Misc. metrics (incl. F1): {report}
  ''')

  if y_prob is not None:
    y_true_bin = (y_true == 'Alienation').astype(int)
    fpr, tpr, _ = roc_curve(y_true_bin, y_prob)
    auc = roc_auc_score(y_true_bin, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
    plt.plot([0, 1], [0, 1], 'k--', label='Random classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.show()

  return report, alpha

In [ ]:
class_report, alpha = get_performance(y_true, y_pred, y_prob)

# Design your own


Now we can practice a bit with designing our own prompt. You can see the different parts of the prompt below and are allowed to edit them in whichever way you please.

A good way to do this is to edit small parts, then run the classification below, inspect the test scores, and repeat the process until you are satisfied.

Let's see how high you can get the validation scores, even if the model is currently not very good.

In [ ]:
sys_prompt = f"""
# Main topic\n\n

You are a professional text coder of TikTok video transcripts about news, politics & societal issues. Your task is to determine whether the given transcript contains group signaling rehtoric, better known as Alienation, or whether it contains some other type of language.\n
follow the category definitions below strictly.
"""

In [ ]:
cat_alienation = f"""
Alienates or implies that another individual or group doesn't belong or fit in, uses us vs. them framing
"""

In [ ]:
cat_other = f"""
The transcript does not contain rhetoric to do with Alienation
"""

In [ ]:
def create_new_prompt(sys_prompt, cat_alienation, cat_other, examples=prompt['examples']) -> None:
  new_prompt = prompt
  new_prompt['system_prompt'] = sys_prompt + '\n\n{categories}'
  new_prompt['categories']['Alienation'] = cat_alienation
  new_prompt['categories']['Other'] = cat_other
  new_prompt['examples'] = examples

  with open('new_prompt.json', 'w') as f:
    json.dump(new_prompt, f)

create_new_prompt(sys_prompt, cat_alienation, cat_other)

In [ ]:
!choicellm transcripts.txt --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'new_prompt.json' > results.csv

results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']
y_prob = results['probs'].str.split(';').str[0].astype(float)

class_report, alpha = get_performance(y_true, y_pred, y_prob)

# Examples

Now, one way you can improve the performance of your prompt drastically is by adding examples to the prompt. Typically you want to inspect your validation set with the predicted labels and see where the model went wrong. The best examples to add are oftentimes transcripts right on the border of two classes (grey area) or things that the model continiously strugles with.

Some potential transcripts you could use as examples can be found in the file 'potential_examples.csv'. This can be downloaded from the GitHub:
https://github.com/Kevinoost1/aps26_llm_ws



In case you would like to add examples, add them in the dictionary format below. the "item" is the transcript and the "target_index" is your label. This is zero-based, meaning target_index: 0 = 'Alienation', and target_index: 1 = 'Other'

Just make sure that your examples do NOT appear in the validation set, and especially NOT the test set either. You can treat this part sort of like a training set in classical machine learning / NLP tasks.

In [ ]:
examples = [
    {
      "item": "what on earth is going on in the house of Commons",
      "target_index": 1
    }
  ]

In [ ]:
create_new_prompt(sys_prompt, cat_alienation, cat_other, examples)

!choicellm transcripts.txt --model 'unsloth/Qwen2.5-7B-Instruct-bnb-4bit' --prompt 'new_prompt.json' > results.csv

results: pd.DataFrame = pd.read_csv('results.csv')
y_pred = results['pred']
y_prob = results['probs'].str.split(';').str[0].astype(float)

class_report, alpha = get_performance(y_true, y_pred, y_prob)

You can from here on out continue to improve all parts of the prompt and add examples as you please.



Remember that this workshop mainly covered the validation part of the process. For a fair assement of your classification performance you need to label a seperate randomly sampled test set and extract performance metrics once with your 'perfect' prompt. This is what you would eventually report.

If you are using an API it might be advisable to run the test classification multiple times and take the average. This is because API-called models are usually the least deterministic in their output.

A further note is that it is possible to correct for your test results in your downstream analyses. If you, for example, wish to run a t-test or regression with your predicted labels, you can look into bootstrapping methods that correct for errors in the model's prediction.

# Helpful resources

**General overview of the methodology and good practices**

- Fang, Q., Bernardo, J. G., & van Kesteren, E. J. (2026). A Methodological Guide on Using Large Language Models for Text Annotation in the Social Sciences and Humanities with Python and R. arXiv preprint arXiv:2604.09638.doi.org/10.48550/arXiv.2604.09638

- Törnberg, P. (2024). Best practices for text annotation with large language models. arXiv preprint arXiv:2402.05129.

**Theoretical paper about LLMs and a small tutorial**

- Hussain, Z., Binz, M., Mata, R., & Wulff, D. U. (2024). A tutorial on open-source large language models for behavioral science. Behavior Research Methods, 56(8), 8214-8237. doi.org/10.3758/s13428-024-02455-8

**If you want to take it a step further - Interesting use of other NLP concepts for psychological research**

- Wulff, D. U., & Mata, R. (2025). Semantic embeddings reveal and address taxonomic incommensurability in psychological measurement. Nature Human Behaviour, 9(5), 944-954.
